In [ ]:
from pathlib import Path
from openai import OpenAI

doc_name = r"datasets\raw\COF_FY24_IP.pdf"

path = Path(doc_name)

file_name = path.name
stem = path.stem
ext = path.suffix
ticker = stem.split("_")[0]
FP = stem.split("_")[1]

print(file_name,stem,ext,ticker,FP)

#Can create loop to iterate through all raw document names. 
# use stem as starting point. 
#extract out ticker and FP



COF_FY24_IP.pdf COF_FY24_IP .pdf COF FY24
ChatCompletionMessage(content='1 + 1 equals 2.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


## Sample Chunking using Langchain Text Splitter, FAISS (non langchain), OpenAI.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import openai
import numpy as np
from pathlib import Path
from openai import OpenAI
import faiss

In [10]:
# Read md file from folder
with open(r"datasets\processed\markdown\COF_FY24_IP.md","r") as f:
    text = f.read()

In [12]:
#Create chunks with recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(text)


In [14]:
def embed_texts(texts, model="text-embedding-3-small"):
    embeds = []
    for t in texts:
        resp = openai.embeddings.create(model=model, input=t)
        embeds.append(resp.data[0].embedding)
    return np.array(embeds)

embeddings = embed_texts(chunks)

In [ ]:
# Dimensions reflect number of chunks, embedding dimensions. 
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

In [ ]:
# Convert query into vector
query_text = "What is the cap rate of COF?"

#Embed texts takes in a list and returns a list of embedded text. We provided a list but only want a single query. 
query_vec = embed_texts([query_text])[0]
#I has (n_queries, k) diemnsions
D, I = index.search(query_vec.reshape(1, -1), k=8)
relevant_chunks = [chunks[i] for i in I[0]]

In [23]:
client = OpenAI()

sample_messages = [
    {"role":"system","content":"You are a financial extractor"},
    {f"role":"user","content":f"Context:\n{relevant_chunks}"},
    {f"role":"user","content":"What is the cap rate of Centuria Office REIT?"}
]

model_name = "gpt-4o-mini"

completion = client.chat.completions.create(messages = sample_messages,model=model_name)

print(completion.choices[0].message)


ChatCompletionMessage(content='The weighted average capitalisation rate (WACR) for Centuria Office REIT (COF) is 6.58%.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [27]:
from pydantic import BaseModel
from pydantic import Field
from typing import Optional

class Metric(BaseModel):
    metric_name_doc: Optional[str] = Field(description="Exact name of metric being queried in document")
    metric_val:Optional[str] = Field(description="Value of metric being extracted")
    metric_period:Optional[str] = Field(description="Financial Period of Metric period extracted. Eg: FYXX, HYXX")



In [ ]:
client = OpenAI()

sample_messages = [
    {"role":"system","content":"You are a financial extractor"},
    {f"role":"user","content":f"Context:\n{relevant_chunks}"},
    {f"role":"user","content":"What is the FY24 cap rate of Centuria Office REIT?"}
]


model_name = "gpt-4o-mini"

#parse is for structured output
completion = client.chat.completions.parse(
    model = model_name,
    messages = sample_messages,
    response_format=Metric
)

## Using Langchain docs to maintain page source

In [41]:
from langchain_core.documents import Document

In [ ]:
docs = []